# 第5回：何を、いつ、何のために予測するか

**今日の問い：モデル構築より前に決めるべきことは何か。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 利用者・判断・予測時点を1文にする
- 目的変数と利用可能な説明変数を分ける
- 業務上意味のあるベースラインと指標を決める

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

### 先に押さえる言葉

- 予測時点：モデルを実際に使う瞬間
- ベースライン：複雑なモデルと比較する単純な基準
- 回帰：連続値を予測する問題
- 分類：クラスやカテゴリを予測する問題

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## 予測問題を1文にする

例：**実験条件を決める時点で利用できる情報から、収率を予測し、優先して実施する条件を選ぶ。**

`post_assay_signal`、`purity_pct`、`yield_pct`は実験後に得られるため、この時点の説明変数にはできません。


In [ ]:
available_at_planning = [
    "scaffold_group", "solvent", "catalyst", "temperature_c", "reaction_time_h",
    "concentration_m", "molecular_weight", "logp", "tpsa", "h_bond_donors", "rotatable_bonds",
]
unavailable_at_planning = ["yield_pct", "active", "post_assay_signal", "purity_pct"]
print("計画時に使える列:", available_at_planning)
print("実験後に得られる列:", unavailable_at_planning)


## TRY：単純な予測を基準にする

複雑なモデルより先に、平均値または最頻値だけを返すモデルを作ります。


In [ ]:
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.model_selection import train_test_split

train, valid = train_test_split(df, test_size=0.25, random_state=42)
reg = DummyRegressor(strategy="mean").fit(train[["molecular_weight"]], train["yield_pct"])
cls = DummyClassifier(strategy="most_frequent").fit(train[["molecular_weight"]], train["active"])
print("平均収率だけで予測したMAE:", round(mean_absolute_error(valid["yield_pct"], reg.predict(valid[["molecular_weight"]])), 2))
print("多数派だけで予測した正解率:", round(accuracy_score(valid["active"], cls.predict(valid[["molecular_weight"]])), 3))


## TRY：自分のテーマを整理する

次の7項目を埋めます。

1. 誰が、何の判断に使うか
2. いつ予測するか
3. 目的変数
4. その時点で利用できる説明変数
5. 利用してはいけない情報
6. 回帰か分類か
7. 単純な基準は何か

## ASK COPILOT

曖昧な点を推測で埋めず、確認質問として返すよう依頼します。


## DEEP DIVE：結果を一段深く読む

次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

### 出力を見る観点

- スコアより先に誰の判断をどう変えるかを確認する
- 未来情報や測定後情報は高性能でも使えない
- 誤りの種類ごとの業務コストを考える


In [ ]:
problem_canvas = pd.DataFrame({
    "項目": ["利用者", "判断", "予測時点", "目的変数", "使える情報", "使えない情報", "評価指標", "単純基準"],
    "例": ["実験担当者", "次に試す条件の優先順位", "実験計画時", "yield_pct", "構造・予定条件", "実験後の測定値", "MAE", "過去平均"],
})
display(problem_canvas)
error_cost = pd.DataFrame({"誤り": ["収率を過大予測", "収率を過小予測"], "起こりうる影響": ["低収率条件へ実験資源を使う", "有望条件を見送る"], "確認したい相手": ["実験担当者", "テーマリーダー"]})
display(error_cost)


## よくある誤り

- 入手できる列をすべて使う
- 目的変数が測定や運用で不安定
- 精度目標だけで利用方法が決まっていない

## SELF-STUDY（任意・30〜60分）

- 自社テーマを機密情報なしで7項目の問題設定へ落とす
- 偽陽性・偽陰性または過大・過小予測のコストを書く

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 誰が何を判断するモデルか
2. 予測時点で本当に得られる列はどれか
3. 単純基準を超えることにどんな価値があるか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
